# 🧠 Optimized MARBERTv2 Clinical Classifier Training (Shifaa 80/20 Split)
### High-Performance Dialectal Arabic Mental Health Diagnosis Pipeline

> **Recommended Runtime:** Go to **Runtime > Change runtime type > Select T4 GPU**.

### 🚀 Why MARBERTv2 for Arabic Mental Health?
1. **Pretrained on 1 Billion Conversational Arabic Texts:** Unlike standard AraBERT (which was trained on formal Wikipedia and news articles), MARBERTv2 was pretrained by UBC NLP on 1B dialectal Arabic expressions across Egyptian, Levantine, Gulf, and Maghrebi dialects.
2. **Superior Emotion & Psychological Idiom Vocabulary:** Patients express distress colloquially ('مخنوق', 'مش طايق نفسي', 'دماغي مش بتسكت', 'ضايقة فيني الوسيعه'). MARBERTv2 has 100,000 subwords to capture these expressions natively.
3. **Expanded Context Window (max_len = 256):** Preserves complete multi-paragraph patient queries.
4. **Balanced Class-Weighted Loss:** Compensates for clinical category imbalance.
5. **Cosine Annealing Scheduler (4 Epochs):** Smooth convergence to maximum generalization.
6. **Top-1 and Top-2 Clinical Comorbidity Evaluation:** Multi-condition psychiatric diagnostic tracking.


In [ ]:
# 1. Install required packages
!pip install -q transformers torch datasets requests matplotlib seaborn accelerate scikit-learn

In [ ]:
# 2. Verify GPU Acceleration
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')

In [ ]:
# 3. Download & Prepare Shifaa Dataset (35,648 Consultations with 80/20 Split)
import os, csv, io, json, random, re, requests
from collections import Counter

CATEGORY_MAPPING = {
    'DEPRESSION': {'keywords': ['اكتئاب', 'حزن', 'كآبة', 'يأس', 'فقدان الشغف', 'إحباط'], 'label_ar': 'أعراض المزاج الاكتئابي والحزن (Depressive Symptoms)', 'specialty': 'CLINICAL_PSYCHOLOGY'},
    'ANXIETY': {'keywords': ['قلق', 'توتر', 'تفكير زائد', 'خوف مستمر', 'عصبية', 'تشتت'], 'label_ar': 'أعراض القلق العام والتوتر (Generalized Anxiety)', 'specialty': 'CBT_SPECIALIST'},
    'PANIC': {'keywords': ['مخاوف', 'هلع', 'رهاب', 'نوبة هلع', 'فزع', 'خفقان'], 'label_ar': 'نوبات الهلع والمخاوف والرهاب (Panic & Phobia)', 'specialty': 'PSYCHIATRY'},
    'SLEEP_DISRUPTION': {'keywords': ['نوم', 'أرق', 'كوابيس', 'سهر', 'استيقاظ متكرر', 'أحلام'], 'label_ar': 'اضطرابات جودة النوم والأرق (Sleep Disruption)', 'specialty': 'CLINICAL_PSYCHOLOGY'},
    'OBSESSIONS_OCD': {'keywords': ['وسواس', 'وساوس', 'قهري', 'شكوك', 'أفكار ملحة'], 'label_ar': 'الوسواس القهري والأفكار المتسلطة (OCD & Obsessions)', 'specialty': 'PSYCHIATRY'},
    'TRAUMA_PTSD': {'keywords': ['صدمة', 'حادث', 'فاجعة', 'كرب', 'ذكريات مؤلمة'], 'label_ar': 'مؤشرات الصدمة النفسية والذكريات الضاغطة (Trauma & PTSD)', 'specialty': 'TRAUMA_PTSD'},
    'FAMILY_CONFLICTS': {'keywords': ['أسرة', 'عائلة', 'زوج', 'زوجة', 'طلاق', 'خلافات أسرية'], 'label_ar': 'الضغوطات والخلافات الأسرية (Family Dynamics)', 'specialty': 'CLINICAL_PSYCHOLOGY'},
    'BURNOUT_STRESS': {'keywords': ['إجهاد', 'ضغط عمل', 'دراسة', 'احتراق', 'إنهاك', 'تطوير الذات'], 'label_ar': 'الإجهاد النفسي والضغوط الحياتية (Burnout & Stress)', 'specialty': 'CBT_SPECIALIST'}
}

SHIFAA_FILES = [
    'Behavioral_Psychological_Conditions.csv',
    'Childhood_Psychological_Problems.csv',
    'Neuropsychological_Conditions.csv',
    'Other_Psychological_and_Behavioral_Issues.csv',
    'Personality_and_Self_Development.csv',
    'Psychological_Issues_and_Guidance.csv',
    'Psychotic_Disorders.csv'
]
HF_BASE_URL = 'https://huggingface.co/datasets/Ahmed-Selem/Shifaa_Arabic_Mental_Health_Consultations/resolve/main/'

def clean_arabic_text(text):
    if not text: return ''
    text = re.sub(r'^(السلام عليكم ورحمة الله وبركاته|مرحبا|تحية طيبة وبعد[،,:]?)\s*', '', text.strip(), flags=re.IGNORECASE)
    text = re.sub(r'(وجزاكم الله خيرا|وشكرا لكم|أفيدوني جزاكم الله خيرا|مع خالص الشكر)[.!]?\s*$', '', text.strip(), flags=re.IGNORECASE)
    text = re.sub(r'[\u064B-\u0652\u0640]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def map_diag(diag_str, text_content):
    diag = diag_str.lower() if diag_str else ''
    text_lower = text_content.lower() if text_content else ''
    if any(k in diag for k in ['اكتئاب', 'حزن', 'كآبة']): return 'DEPRESSION'
    if any(k in diag for k in ['هلع', 'مخاوف', 'رهاب', 'فزع']): return 'PANIC'
    if any(k in diag for k in ['نوم', 'أرق', 'كابوس']): return 'SLEEP_DISRUPTION'
    if any(k in diag for k in ['وسواس', 'قهري']): return 'OBSESSIONS_OCD'
    if any(k in diag for k in ['قلق', 'توتر']): return 'ANXIETY'
    if any(k in diag for k in ['صدم', 'كرب']): return 'TRAUMA_PTSD'
    if any(k in diag for k in ['أسر', 'عائل', 'زوج']): return 'FAMILY_CONFLICTS'
    for cat, meta in CATEGORY_MAPPING.items():
        if any(k in text_lower for k in meta['keywords']):
            return cat
    return 'BURNOUT_STRESS'

print('Downloading and parsing Shifaa dataset files...')
all_data = []
for fn in SHIFAA_FILES:
    r = requests.get(HF_BASE_URL + fn)
    reader = csv.DictReader(io.StringIO(r.content.decode('utf-8', errors='ignore')))
    for row in reader:
        q = clean_arabic_text(row.get('Question', ''))
        t = clean_arabic_text(row.get('Question Title', ''))
        d = row.get('Hierarchical Diagnosis', '')
        if len(q) < 20: continue
        # Enhanced clinical anchor formatting for attention layers
        full = f"استشارة سريرية: {t} | تفاصيل شكوى المريض: {q}" if t else q
        all_data.append({'text': full, 'category': map_diag(d, full)})

print(f'Total Consultations Loaded: {len(all_data)}')

# 80/20 Stratified Split
os.makedirs('data', exist_ok=True)
categories = sorted(list(CATEGORY_MAPPING.keys()))
label2id = {c: i for i, c in enumerate(categories)}
id2label = {i: c for i, c in enumerate(categories)}

with open('data/label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump({'label2id': label2id, 'id2label': id2label, 'category_details': CATEGORY_MAPPING}, f, ensure_ascii=False, indent=2)

by_cat = {}
for item in all_data: by_cat.setdefault(item['category'], []).append(item)

train_rows, test_rows = [], []
random.seed(42)
for cat, items in by_cat.items():
    random.shuffle(items)
    idx = int(len(items) * 0.8)
    train_rows.extend(items[:idx])
    test_rows.extend(items[idx:])

random.shuffle(train_rows)
random.shuffle(test_rows)

with open('data/train.jsonl', 'w', encoding='utf-8') as f:
    for r in train_rows: f.write(json.dumps({'text': r['text'], 'label': label2id[r['category']]}, ensure_ascii=False) + '\n')
with open('data/test.jsonl', 'w', encoding='utf-8') as f:
    for r in test_rows: f.write(json.dumps({'text': r['text'], 'label': label2id[r['category']]}, ensure_ascii=False) + '\n')

print(f'Train size (80%): {len(train_rows)} | Test size (20%): {len(test_rows)}')

In [ ]:
# 4. PyTorch Dataset & DataLoader with max_len = 256
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_cosine_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

MODEL_NAME = 'UBC-NLP/MARBERTv2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 256 tokens captures complete multi-paragraph psychiatric inquiries
MAX_LEN = 256

class TextDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_len=MAX_LEN):
        self.samples = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                d = json.loads(line)
                self.samples.append((d['text'], d['label']))
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        t, l = self.samples[i]
        enc = self.tokenizer(t, truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0), 'label': torch.tensor(l, dtype=torch.long)}

train_ds = TextDataset('data/train.jsonl', tokenizer)
test_ds = TextDataset('data/test.jsonl', tokenizer)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
print(f'DataLoaders ready with max_len={MAX_LEN}.')

In [ ]:
# 5. Training AraBERT with Class-Weighted Loss & Cosine Scheduler
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
).to(device)

# Compute Balanced Class Weights
train_labels_list = [s[1] for s in train_ds.samples]
class_weights = compute_class_weight('balanced', classes=np.arange(len(label2id)), y=train_labels_list)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights_tensor)

EPOCHS = 4
optimizer = torch.optim.AdamW(model.parameters(), lr=2.5e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

best_acc = 0.0
best_top2 = 0.0
os.makedirs('arabert_classifier', exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    t0 = time.time()
    for step, b in enumerate(train_loader):
        input_ids, mask, labels = b['input_ids'].to(device), b['attention_mask'].to(device), b['label'].to(device)
        optimizer.zero_grad()
        out = model(input_ids=input_ids, attention_mask=mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if (step + 1) % 300 == 0:
            print(f'Epoch {epoch}/{EPOCHS} | Step {step+1}/{len(train_loader)} | Loss: {total_loss/(step+1):.4f}')
    
    # Validation on 20% test set (Top-1 and Top-2 Accuracy)
    model.eval()
    preds, trues = [], []
    top2_hits = 0
    with torch.no_grad():
        for b in test_loader:
            input_ids, mask, labels = b['input_ids'].to(device), b['attention_mask'].to(device), b['label'].to(device)
            out = model(input_ids=input_ids, attention_mask=mask)
            preds.extend(torch.argmax(out.logits, dim=1).cpu().tolist())
            trues.extend(labels.cpu().tolist())
            
            # Top-2 Evaluation (Comorbidity Accuracy)
            top2_indices = torch.topk(out.logits, k=2, dim=1).indices.cpu().tolist()
            for t, p2 in zip(labels.cpu().tolist(), top2_indices):
                if t in p2: top2_hits += 1
    
    acc = sum(1 for p, t in zip(preds, trues) if p == t) / len(trues)
    top2_acc = top2_hits / len(trues)
    print(f'\n🌟 Epoch {epoch} Complete in {time.time()-t0:.1f}s | Top-1 Acc: {acc*100:.2f}% | Top-2 Acc: {top2_acc*100:.2f}%')
    if acc > best_acc:
        best_acc = acc
        best_top2 = top2_acc
        model.save_pretrained('arabert_classifier')
        tokenizer.save_pretrained('arabert_classifier')
        print('⭐ Saved New Best Checkpoint!')

In [ ]:
# 6. Generate Confusion Matrix & Classification Report Plot
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

target_names = [id2label[i] for i in range(len(id2label))]
print('\n' + '='*65)
print(f'Final Test Evaluation (Top-1: {best_acc*100:.2f}% | Top-2 Comorbidity: {best_top2*100:.2f}%):')
print('='*65)
print(classification_report(trues, preds, target_names=target_names))

cm = confusion_matrix(trues, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title(f'MARBERTv2 Confusion Matrix (Top-1 Acc: {best_acc*100:.1f}% | Top-2 Acc: {best_top2*100:.1f}%)', fontsize=13)
plt.xlabel('Predicted Diagnosis', fontsize=12)
plt.ylabel('Ground Truth Diagnosis', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
# 7. Zip & Download Trained Model Checkpoint
!zip -r arabert_classifier.zip arabert_classifier/ confusion_matrix.png data/label_mapping.json
from google.colab import files
files.download('arabert_classifier.zip')